In [5]:
!pip install pandas numpy scikit-learn xgboost openpyxl joblib



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Load Dataset
# Ensure "health_nutrition_disease_dataset_12000.xlsx" is in the same folder
file_path = "health_nutrition_disease_dataset_12000.xlsx"
try:
    df = pd.read_excel(file_path)
    print(f"Dataset loaded: {df.shape}")
except FileNotFoundError:
    print("Error: The Excel file was not found. Please check the file path.")

# 2. Data Preprocessing
# Encoding Gender and Disease Risks
df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})

disease_columns = [
    "Diabetes_Risk", "Hypertension_Risk", "Heart_Disease_Risk",
    "Obesity_Risk", "Anemia_Risk", "Kidney_Disease_Risk"
]

for col in disease_columns:
    df[col] = df[col].map({"High": 1, "Low": 0})

# Define Features
features = [
    "Age", "Gender", "BMI", "Daily_Calories_kcal", "Carbohydrates_g",
    "Protein_g", "Total_Fat_g", "Saturated_Fat_g", "Trans_Fat_g",
    "Total_Sugar_g", "Added_Sugar_g", "Fiber_g", "Sodium_mg",
    "Potassium_mg", "Calcium_mg", "Iron_mg", "Vitamin_D_IU",
    "Vitamin_B12_mcg", "Physical_Activity_min", "Water_Intake_L"
]

X = df[features]

# 3. Scaling & Train-Test Split
X_train_raw, X_test_raw = train_test_split(X, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Save the scaler for future use
joblib.dump(scaler, "scaler.pkl")

# 4. Training Multi-Output Models
models = {}
print("\n--- Training Results ---")

for disease in disease_columns:
    y = df[disease]
    y_train, y_test = train_test_split(y, test_size=0.2, random_state=42)

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, preds)
    print(f"{disease.replace('_', ' ')} accuracy: {acc:.3f}")

    # Save each model
    models[disease] = model
    joblib.dump(model, f"{disease}_model.pkl")

print("\nModels and Scaler saved successfully.")

# 5. Prediction Logic for New Patient
def calculate_bmi(weight_kg, height_cm):
    height_m = height_cm / 100
    return round(weight_kg / (height_m ** 2), 1)

# Example Patient Data
patient = {
    "Age": 45,
    "Gender": 1,  # 1 = Female
    "Weight_kg": 70,
    "Height_cm": 154,
    "Daily_Calories_kcal": 2800,
    "Carbohydrates_g": 350,
    "Protein_g": 90,
    "Total_Fat_g": 130,
    "Saturated_Fat_g": 45,
    "Trans_Fat_g": 2,
    "Total_Sugar_g": 120,
    "Added_Sugar_g": 90,
    "Fiber_g": 15,
    "Sodium_mg": 3500,
    "Potassium_mg": 2000,
    "Calcium_mg": 500,
    "Iron_mg": 7,
    "Vitamin_D_IU": 2190,
    "Vitamin_B12_mcg": 1.8,
    "Physical_Activity_min": 20,
    "Water_Intake_L": 1.2
}

# Process patient data for prediction
patient["BMI"] = calculate_bmi(patient["Weight_kg"], patient["Height_cm"])
patient_df = pd.DataFrame([{k: patient[k] for k in features}])
patient_scaled = scaler.transform(patient_df)

# 6. Final Risk Output
print("\n--- Individual Risk Prediction ---")
for disease in disease_columns:
    pred = models[disease].predict(patient_scaled)[0]
    risk_status = " High Risk" if pred == 1 else " Low Risk"
    print(f"{disease.replace('_', ' '):<20}: {risk_status}")

Dataset loaded: (12000, 26)

--- Training Results ---
Diabetes Risk accuracy: 0.999
Hypertension Risk accuracy: 0.999
Heart Disease Risk accuracy: 1.000
Obesity Risk accuracy: 0.999
Anemia Risk accuracy: 1.000
Kidney Disease Risk accuracy: 1.000

Models and Scaler saved successfully.

--- Individual Risk Prediction ---
Diabetes Risk       : ✅ Low Risk
Hypertension Risk   : ⚠️ High Risk
Heart Disease Risk  : ⚠️ High Risk
Obesity Risk        : ✅ Low Risk
Anemia Risk         : ⚠️ High Risk
Kidney Disease Risk : ⚠️ High Risk
